In [2]:
%%pyspark

# Load source datasets into DataFrames for profiling
df_stores_data = spark.read.csv("Files/bronze/stores_products_data/Stores_Catalog.csv", header=True, inferSchema=True)
df_products_data = spark.read.csv("Files/bronze/stores_products_data/Products_Master.csv", header=True, inferSchema=True)
df_invoices_header = spark.read.format("delta").load("Files/bronze/pos_retail_transactions/Invoices_Header")
df_invoices_lines = spark.read.format("delta").load("Files/bronze/pos_retail_transactions/Invoices_Lines")
print("Source DataFrames loaded successfully")

StatementMeta(, 87c83071-b0d4-49a2-ad9a-6bf29304a7de, 4, Finished, Available, Finished, False)

Source DataFrames loaded successfully


In [ ]:
%%pyspark

# Create temporary SQL views for Spark SQL profiling
df_stores_data.createOrReplaceTempView("stores")
df_products_data.createOrReplaceTempView("products")
df_invoices_header.createOrReplaceTempView("invoices_header")
df_invoices_lines.createOrReplaceTempView("invoices_lines")
print("Temporary SQL views created successfully")

StatementMeta(, 87c83071-b0d4-49a2-ad9a-6bf29304a7de, 5, Finished, Available, Finished, False)

Temporary SQL views created successfully


In [3]:
-- Data Profiling for stores
    -- Show sample data
SELECT * FROM stores LIMIT 10;
    -- Profile completeness, uniqueness, and basic validity
SELECT 
    count(*) AS total_records,
    SUM(CASE WHEN StoreID IS NULL THEN 1 ELSE 0 END) AS total_null_store_id,
    count(distinct(StoreID)) AS unique_store_id,
    SUM(CASE WHEN StoreID <= 0 THEN 1 ELSE 0 END) AS total_invalid_store_id,
    SUM(CASE WHEN StoreName IS NULL OR TRIM(StoreName) = '' THEN 1 ELSE 0 END) AS total_missing_store_name,
    count(distinct(StoreName)) AS unique_store_name,
    SUM(CASE WHEN Region IS NULL OR TRIM(Region) = '' THEN 1 ELSE 0 END) AS total_missing_region,
    count(distinct(Region)) AS unique_region,
    SUM(CASE WHEN City IS NULL OR TRIM(City) = '' THEN 1 ELSE 0 END) AS total_missing_city,
    count(distinct(City)) AS unique_city
FROM stores;
    -- Inspect categorical values for unexpected values
SELECT DISTINCT(Region) AS unique_regions FROM stores;

SELECT DISTINCT(City) AS unique_cities FROM stores;

StatementMeta(, 04f22d33-598d-479f-aa80-00342021857a, 8, Finished, Available, Finished, True)

<Spark SQL result set with 10 rows and 4 fields>

<Spark SQL result set with 1 rows and 10 fields>

<Spark SQL result set with 4 rows and 1 fields>

<Spark SQL result set with 5 rows and 1 fields>

In [5]:
%%sql

-- Data Profiling for products
    -- Show sample data
SELECT * FROM products LIMIT 5;
    -- Main profiling query
SELECT 
    COUNT(*) AS total_records,
    SUM(CASE WHEN ProductID IS NULL THEN 1 ELSE 0 END) AS total_null_product_id,
    COUNT(DISTINCT(ProductID)) AS unique_product_id,
    SUM(CASE WHEN ProductID <= 0 THEN 1 ELSE 0 END) AS total_invalid_product_id ,
    SUM(CASE WHEN ProductName IS NULL OR TRIM(ProductName) = '' THEN 1 ELSE 0 END) AS total_missing_product_name,
    COUNT(DISTINCT(ProductName)) AS unique_product_name,
    SUM(CASE WHEN Category IS NULL OR TRIM(Category) = '' THEN 1 ELSE 0 END) AS total_missing_category,
    COUNT(DISTINCT(Category)) AS unique_category,
    SUM(CASE WHEN UnitPrice IS NULL THEN 1 ELSE 0 END) AS total_missing_unit_price,
    SUM(CASE WHEN UnitPrice <= 0 THEN 1 ELSE 0 END) AS total_invalid_unit_price,
    COUNT(DISTINCT(UnitPrice)) AS unique_unit_price,
    MAX(UnitPrice) AS maximum_price,
    MIN(UnitPrice) AS minimum_price
FROM products;
    -- Inspect categorical values and detect suspicious
SELECT DISTINCT(Category) FROM products;
    -- Check the count of matching ProductID and UnitPrice together
SELECT COUNT(*)
FROM invoices_lines as il 
INNER JOIN products as p
ON il.ProductID = p.ProductID and il.UnitPrice = p.UnitPrice
UNION ALL 
SELECT COUNT(*) FROM invoices_lines

StatementMeta(, 04f22d33-598d-479f-aa80-00342021857a, 16, Finished, Available, Finished, True)

<Spark SQL result set with 5 rows and 4 fields>

<Spark SQL result set with 1 rows and 13 fields>

<Spark SQL result set with 4 rows and 1 fields>

<Spark SQL result set with 2 rows and 1 fields>

In [8]:
%%sql

-- Data Profiling for invoices header
    -- Show sample data
SELECT * FROM invoices_header LIMIT 5;
    -- Main profiling query
SELECT 
    COUNT(*) AS total_count,
    COUNT(*) - COUNT(InvoiceID) As total_null_invoice_id,
    COUNT(DISTINCT InvoiceID) AS unique_invoice_id,
    COUNT(*) - COUNT(StoreID) As total_null_store_id,
    COUNT(DISTINCT StoreID) AS unique_store_id,
    SUM(CASE WHEN StoreID <=0 THEN 1 ELSE 0 END) AS invalid_store_id,
    COUNT(*) - COUNT(InvoiceDate) As total_null_invoice_date,
    MAX(InvoiceDate) AS maximum_date,
    MIN(InvoiceDate) AS minimum_date,
    SUM(CASE WHEN PaymentMethod IS NULL OR TRIM(PaymentMethod) = '' THEN 1 ELSE 0 END) As total_missing_payment_method,
    COUNT(DISTINCT PaymentMethod) AS unique_payment_method,
    SUM(CASE WHEN OrderStatus IS NULL OR TRIM(OrderStatus) = '' THEN 1 ELSE 0 END) As total_missing_order_status,
    COUNT(DISTINCT OrderStatus) AS unique_order_status
FROM invoices_header;
    -- Inspect categorical values and detect suspicious
SELECT 
    DISTINCT(PaymentMethod) AS unique_payment_method
FROM invoices_header;

SELECT 
    OrderStatus AS unique_order_status,
    COUNT(*) AS total_number
FROM invoices_header
GROUP BY OrderStatus;
    -- Validate LastModifiedAt baseline
SELECT
SUM(CASE WHEN InvoiceDate <> LastModifiedAt THEN 1 ELSE 0 END) AS total_changed_status
FROM invoices_header;

StatementMeta(, 04f22d33-598d-479f-aa80-00342021857a, 29, Finished, Available, Finished, True)

<Spark SQL result set with 5 rows and 6 fields>

<Spark SQL result set with 1 rows and 13 fields>

<Spark SQL result set with 3 rows and 1 fields>

<Spark SQL result set with 3 rows and 2 fields>

<Spark SQL result set with 1 rows and 1 fields>

In [10]:
%%sql

-- Data Profiling for invoices lines
    -- Show sample data
SELECT * FROM invoices_lines LIMIT 5;
    -- Main profiling query
SELECT
    COUNT(*) AS total_records,
    COUNT(*) - COUNT(LineID) AS total_null_line_id,
    COUNT(DISTINCT LineID) AS unique_line_id,
    COUNT(*) - COUNT(InvoiceID) AS total_null_invoice_id,
    COUNT(DISTINCT InvoiceID) AS unique_invoice_id,
    COUNT(*) - COUNT(ProductID) AS total_null_product_id,
    COUNT(DISTINCT ProductID) AS unique_product_id,
    SUM(CASE WHEN ProductID <= 0 THEN 1 ELSE 0 END) AS total_invalid_product_id,
    COUNT(*) - COUNT(Quantity) AS total_null_quantity,
    MIN(Quantity) AS minimum_quantity,
    MAX(Quantity) AS maximum_quantity,
    SUM(CASE WHEN Quantity <= 0 THEN 1 ELSE 0 END) total_invalid_quantity,
    COUNT(*) - COUNT(UnitPrice) AS total_null_unit_price,
    COUNT(DISTINCT UnitPrice) AS unique_unit_price,
    SUM(CASE WHEN UnitPrice <= 0 THEN 1 ELSE 0 END) total_invalid_unit_price
FROM invoices_lines;
    -- Business impact: invoices with no analytically valid lines
SELECT COUNT(*) AS invoices_without_valid_lines
FROM invoices_header AS h
LEFT JOIN invoices_lines AS l
ON l.InvoiceID = h.InvoiceID 
    AND l.Quantity IS NOT NULL
WHERE l.InvoiceID IS NULL

StatementMeta(, 04f22d33-598d-479f-aa80-00342021857a, 38, Finished, Available, Finished, True)

<Spark SQL result set with 5 rows and 5 fields>

<Spark SQL result set with 1 rows and 15 fields>

<Spark SQL result set with 1 rows and 1 fields>

In [9]:
%%sql
-- Referential Integrity & Relationship Coverage Checks

-- Header and Lines relation check
    -- Referential Integrity:
    -- Breaks referential integrity if the result not equal zero 
SELECT COUNT(*) AS number_of_lines_without_header
FROM invoices_lines as il
LEFT JOIN  invoices_header as ih
ON ih.InvoiceID = il.InvoiceID
WHERE ih.InvoiceID IS NULL;
    -- Business observation:
    -- Its not allowed to be header without lines
SELECT COUNT(*) AS number_of_header_without_lines
FROM invoices_header as ih
LEFT JOIN invoices_lines as il
ON ih.InvoiceID = il.InvoiceID
WHERE il.InvoiceID IS NULL;

-- Stores and Header relation check 
    -- Referential Integrity:
    -- Breaks referential integrity if the result not equal zero 
SELECT COUNT(*) AS number_of_headers_without_stores
FROM invoices_header AS ih
LEFT JOIN stores as s
ON s.StoreID = ih.StoreID
WHERE s.StoreID IS NULL;
    -- Business observation:
    -- There may be StoreID that not used in Header 
SELECT COUNT(*) AS number_of_stores_without_headers
FROM stores as s
LEFT JOIN invoices_header as ih
ON s.StoreID = ih.StoreID
WHERE ih.StoreID IS NULL;

-- Products Lines relation check
    -- Referential Integrity:
    -- Breaks referential integrity if the result is not zero
SELECT COUNT(*) AS number_of_lines_without_product
FROM  invoices_lines AS il
LEFT JOIN products AS p
ON il.ProductID = p.ProductID
WHERE p.ProductID IS NULL;    
    -- Business observation:
    -- A product may legitimately have no sales during the analyzed period
SELECT COUNT(*) AS number_of_products_without_lines
FROM products AS p 
LEFT JOIN invoices_lines AS il
ON p.ProductID = il.ProductID
WHERE il.ProductID IS NULL;


StatementMeta(, 04f22d33-598d-479f-aa80-00342021857a, 35, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>